# Cityscapes: полный запуск эксперимента в Google Colab

Notebook только настраивает окружение и вызывает скрипты проекта. Dataset, обучение, метрики и оценка остаются в `src/`. Cityscapes загружается через `kagglehub.dataset_download("electraawais/cityscape-dataset")`. Перед запуском выберите **Runtime → Change runtime type → GPU**.

## 1. Проверка GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU не найден. Выберите Runtime → Change runtime type → GPU.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
!nvidia-smi

## 2. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub

path = kagglehub.dataset_download("electraawais/cityscape-dataset")
print('KaggleHub dataset:', path)

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/atikkhon/diploma.git"
CITYSCAPES_ROOT = Path(path).expanduser().resolve()

PROJECT_DIR = Path('/content/Diploma')
DRIVE_ROOT = Path('/content/drive/MyDrive/cityscapes_robustness')
LOG_DIR = DRIVE_ROOT / 'logs'
STAGE_DIR = DRIVE_ROOT / 'stage_markers'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
TRAINING_METRICS_DIR = DRIVE_ROOT / 'training_metrics'
MLFLOW_DB = DRIVE_ROOT / 'mlflow.db'
MLFLOW_ARTIFACT_DIR = DRIVE_ROOT / 'mlartifacts'
SPLIT_MANIFEST = DRIVE_ROOT / 'split_manifest.csv'
RUNTIME_CONFIG = DRIVE_ROOT / 'experiment_colab.yaml'

for directory in (DRIVE_ROOT, LOG_DIR, STAGE_DIR, CHECKPOINT_DIR, TRAINING_METRICS_DIR, MLFLOW_ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    'REPO_URL': REPO_URL,
    'PROJECT_DIR': str(PROJECT_DIR),
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'LOG_DIR': str(LOG_DIR),
    'STAGE_DIR': str(STAGE_DIR),
    'CHECKPOINT_DIR': str(CHECKPOINT_DIR),
    'SPLIT_MANIFEST': str(SPLIT_MANIFEST),
    'RUNTIME_CONFIG': str(RUNTIME_CONFIG),
    'MLFLOW_TRACKING_URI': f'sqlite:///{MLFLOW_DB}',
    'MLFLOW_ARTIFACT_ROOT': MLFLOW_ARTIFACT_DIR.resolve().as_uri(),
})
print('Drive artifacts:', DRIVE_ROOT)
print('MLflow tracking URI:', os.environ['MLFLOW_TRACKING_URI'])
print('MLflow artifact root:', os.environ['MLFLOW_ARTIFACT_ROOT'])

## 3. Клонирование или обновление Git-репозитория

При повторном запуске выполняется `git pull --ff-only`; локальные изменения в `/content/Diploma` не перезаписываются.

In [ ]:
%%bash
set -euo pipefail
if [[ -d "$PROJECT_DIR/.git" ]]; then
  git -C "$PROJECT_DIR" pull --ff-only 2>&1 | tee -a "$LOG_DIR/git_update.log"
else
  git clone "$REPO_URL" "$PROJECT_DIR" 2>&1 | tee -a "$LOG_DIR/git_update.log"
fi

## 4. Установка `requirements.txt`

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pip install -r requirements.txt 2>&1 | tee -a "$LOG_DIR/install.log"

### Проверка SQLite MLflow

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.tracking import check_mlflow_connection

mlflow_info = check_mlflow_connection('cityscapes_robustness')
print(mlflow_info)

## 5. Пути Cityscapes и Colab-конфигурация

Manifest, checkpoint, история обучения, логи и MLflow переживают разрыв runtime, потому что сохраняются в Google Drive.

In [ ]:
import sys
import yaml

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.dataset import discover_cityscapes_layout, prepare_train_id_masks

layout = discover_cityscapes_layout(CITYSCAPES_ROOT)
# Маски читаются на каждой эпохе, поэтому держим кэш на локальном SSD Colab.
# После обрыва runtime эта ячейка быстро восстановит его из Kaggle labelIds.
prepared_gtfine = Path('/content/prepared_gtFine')
train_masks = prepare_train_id_masks(
    layout['train_masks'], prepared_gtfine / 'train'
)
val_masks = prepare_train_id_masks(
    layout['val_masks'], prepared_gtfine / 'val'
)

with (PROJECT_DIR / 'configs/experiment.yaml').open(encoding='utf-8') as file:
    config = yaml.safe_load(file)
config['data']['root'] = str(CITYSCAPES_ROOT)
config['data']['train_images'] = str(layout['train_images'])
config['data']['train_masks'] = str(train_masks)
config['data']['official_val_images'] = str(layout['val_images'])
config['data']['official_val_masks'] = str(val_masks)
config['data']['split_file'] = str(SPLIT_MANIFEST)
config['training']['checkpoint_dir'] = str(CHECKPOINT_DIR)
config['training']['history_dir'] = str(TRAINING_METRICS_DIR)
config['training']['log_interval'] = 25
config['evaluation']['metrics_dir'] = str(PROJECT_DIR / 'outputs/metrics')
config['evaluation']['predictions_dir'] = str(PROJECT_DIR / 'outputs/predictions')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)
print('Runtime config:', RUNTIME_CONFIG)
print('Train images:', layout['train_images'])
print('Train masks:', train_masks)
print('Official val images:', layout['val_images'])
print('Official val masks:', val_masks)

## 6. Создание `split_manifest.csv`

Manifest пересоздаётся детерминированно, чтобы в нём всегда были актуальные локальные пути. При повторном запуске полная проверка масок не дублируется.

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -s "$SPLIT_MANIFEST" ]]; then
  echo "Обновление путей существующего manifest без повторной полной проверки масок" | tee -a "$LOG_DIR/create_split.log"
  python scripts/create_split.py --config "$RUNTIME_CONFIG" --skip-mask-validation 2>&1 | tee -a "$LOG_DIR/create_split.log"
else
  python scripts/create_split.py --config "$RUNTIME_CONFIG" 2>&1 | tee -a "$LOG_DIR/create_split.log"
fi

## 7. Запуск тестов

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -v 2>&1 | tee -a "$LOG_DIR/pytest.log"

## 8. Dataset smoke test

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/smoke_test_dataset.py --config "$RUNTIME_CONFIG" --split dev 2>&1 | tee -a "$LOG_DIR/dataset_smoke_test.log"

In [ ]:
from IPython.display import Image, display
smoke_png = PROJECT_DIR / 'outputs/figures/dataset_smoke_test.png'
if not smoke_png.is_file():
    raise FileNotFoundError(f'Smoke-test PNG не найден: {smoke_png}')
display(Image(filename=str(smoke_png)))

## Выбор отдельной модульной версии

Первые восемь разделов используют стабильную `main`. Теперь переключите локальный Colab-клон на отдельную ветку U-Net и проверьте её тестами.

In [ ]:
%%bash
set -euo pipefail
git -C "$PROJECT_DIR" fetch origin codex/unet-modular-pipeline
git -C "$PROJECT_DIR" switch codex/unet-modular-pipeline
git -C "$PROJECT_DIR" pull --ff-only
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -q

## 9. Параметры независимого запуска U-Net

Задайте уникальное `RUN_NAME`. Для новой тренировки используйте `RESUME_TRAINING = False`; для продолжения того же запуска — то же имя и `True`. В будущем у каждой модели будет собственный блок в `MODEL_SETTINGS`.

In [ ]:
import os
import yaml

SELECTED_MODEL = 'unet'
RUN_NAME = 'unet_experiment_01'
RESUME_TRAINING = False

MODEL_SETTINGS = {
    'unet': {
        'model': {
            'name': 'unet',
            'encoder_name': 'resnet34',
            'encoder_weights': 'imagenet',
        },
        'training': {
            'epochs': 8,
            'batch_size': 8,
            'num_workers': 2,
            'log_interval': 25,
            'learning_rate': 0.0003,
            'weight_decay': 0.0001,
            'device': 'auto',
            'mixed_precision': True,
        },
        'evaluation': {
            'batch_size': 8,
        },
    },
}

selected = MODEL_SETTINGS[SELECTED_MODEL]
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
RUN_CONFIG = RUN_DIR / 'run_config.yaml'
RUN_DIR.mkdir(parents=True, exist_ok=True)

with RUNTIME_CONFIG.open(encoding='utf-8') as file:
    run_config = yaml.safe_load(file)
run_config['run'] = {'name': RUN_NAME, 'output_dir': str(RUN_DIR)}
run_config['model'] = selected['model']
run_config['training'].update(selected['training'])
run_config['evaluation'].update(selected['evaluation'])
run_config['tracking']['experiment_name'] = 'cityscapes_unet'
run_config['corruptions'] = {
    'darkness': {
        'levels': {
            1: {'factor': 0.75},
            2: {'factor': 0.55},
            3: {'factor': 0.35},
        }
    }
}

with RUN_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(run_config, file, allow_unicode=True, sort_keys=False)

os.environ['RUN_CONFIG'] = str(RUN_CONFIG)
os.environ['RUN_NAME'] = RUN_NAME
os.environ['RESUME_TRAINING'] = '1' if RESUME_TRAINING else '0'
print('Run:', RUN_NAME)
print('Config:', RUN_CONFIG)
print('Resume:', RESUME_TRAINING)

## 10. Обучение или resume выбранного запуска

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print('GPU cache очищен')

## 11. Визуализация clean-сегментации

Задайте индекс изображения internal dev.

In [ ]:
IMAGE_INDEX = 0
os.environ['IMAGE_INDEX'] = str(IMAGE_INDEX)

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition clean 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_clean.log"

In [ ]:
clean_preview = RUN_DIR / 'figures' / f'segmentation_clean_index_{IMAGE_INDEX}.png'
display(Image(filename=str(clean_preview)))

## 12. Clean evaluation выбранного запуска

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"

## 13. Darkness evaluation

Выберите вручную уровень 1, 2 или 3. Каждая оценка добавляется в CSV и создаёт отдельный дочерний run в MLflow.

In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY должен быть 1, 2 или 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"

## 14. Визуализация сегментации после darkness

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"

In [ ]:
darkness_preview = RUN_DIR / 'figures' / f'segmentation_darkness_s{DARKNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(darkness_preview)))

## 15. Все сохранённые результаты

In [ ]:
import pandas as pd

display(pd.read_csv(RUN_DIR / 'metrics' / 'training_history.csv'))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

## 16. MLflow UI

Запустите ячейку один раз. В интерфейсе родительский run содержит обучение и preview, дочерние runs — отдельные clean/darkness оценки.

In [ ]:
import subprocess
from google.colab import output

mlflow_process = subprocess.Popen([
    'mlflow', 'server',
    '--backend-store-uri', os.environ['MLFLOW_TRACKING_URI'],
    '--host', '0.0.0.0',
    '--port', '5000',
])
output.serve_kernel_port_as_iframe(5000, height=800)

## Повторный запуск после обрыва runtime

1. Повторите разделы 1–8.
2. В разделе 9 укажите прежний `RUN_NAME` и `RESUME_TRAINING = True`.
3. Запустите обучение: оно продолжится из `last.pt` и того же MLflow run.
4. Для совершенно нового эксперимента задайте новое `RUN_NAME` и `RESUME_TRAINING = False`.
5. Clean и darkness evaluation можно запускать многократно: каждая оценка сохраняется отдельно.